# Presentation

```{important} AI Disclosure
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

In [1]:
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.linear_model import Ridge
from scipy.stats import loguniform, randint, uniform
from sklearn.linear_model import Lasso
from sklearn.manifold import TSNE
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
df = pd.read_csv("../data/box_office_data_analysis.csv")
df

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,worldwideGross,worldwideGrossCurrency,productionBudget,productionBudgetCurrency,totalStarMeter,genres_list,grossBudgetRatio,grossBudgetRatio_log,genres_parsed,cluster
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"['Adventure', 'Comedy', 'Sci-Fi']","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...,433030505.0,USD,200000000.0,USD,37,"['Adventure', 'Comedy', 'Sci-Fi']",2.165153,0.772491,"['Adventure', 'Comedy', 'Sci-Fi']",0
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"['Animation', 'Adventure', 'Comedy', 'Family',...","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo...",437751829.0,USD,110000000.0,USD,66,"['Animation', 'Adventure', 'Comedy', 'Family',...",3.979562,1.381172,"['Animation', 'Adventure', 'Comedy', 'Family',...",0
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"['Crime', 'Drama', 'Thriller']","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco...",72559167.0,USD,90000000.0,USD,49,"['Crime', 'Drama', 'Thriller']",0.806213,-0.215407,"['Crime', 'Drama', 'Thriller']",0
3,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,7860.0,"['Drama', 'Mystery', 'Thriller']","{'aggregateRating': 6.8, 'voteCount': 128853}",A struggling young woman is relieved by the ch...,399318859.0,USD,35000000.0,USD,31,"['Drama', 'Mystery', 'Thriller']",11.409110,2.434412,"['Drama', 'Mystery', 'Thriller']",0
4,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"['Action', 'Adventure', 'Comedy']","{'aggregateRating': 5.6, 'voteCount': 52429}",A group of friends are going through a mid-lif...,134974943.0,USD,45000000.0,USD,37,"['Action', 'Adventure', 'Comedy']",2.999443,1.098427,"['Action', 'Adventure', 'Comedy']",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,tt0119081,movie,Event Horizon,Event Horizon,{'url': 'https://m.media-amazon.com/images/M/M...,1997.0,5760.0,"['Horror', 'Sci-Fi', 'Thriller']","{'aggregateRating': 6.6, 'voteCount': 214320}",A rescue crew is tasked with investigating the...,26677592.0,USD,60000000.0,USD,6,"['Horror', 'Sci-Fi', 'Thriller']",0.444627,-0.810521,"['Horror', 'Sci-Fi', 'Thriller']",1
739,tt0374900,movie,Napoleon Dynamite,Napoleon Dynamite,{'url': 'https://m.media-amazon.com/images/M/M...,2004.0,5760.0,['Comedy'],"{'aggregateRating': 7, 'voteCount': 255418}",A listless and alienated teenager helps his ne...,46141106.0,USD,400000.0,USD,0,['Comedy'],115.352765,4.747995,['Comedy'],1
740,tt0095765,movie,Cinema Paradiso,Nuovo Cinema Paradiso,{'url': 'https://m.media-amazon.com/images/M/M...,1988.0,10440.0,"['Drama', 'Romance']","{'aggregateRating': 8.5, 'voteCount': 316663}","Salvatore, a famous film director, returns to ...",13502484.0,USD,5000000.0,USD,0,"['Drama', 'Romance']",2.700497,0.993436,"['Drama', 'Romance']",0
741,tt1226837,movie,Beautiful Boy,Beautiful Boy,{'url': 'https://m.media-amazon.com/images/M/M...,2018.0,7200.0,"['Biography', 'Drama']","{'aggregateRating': 7.4, 'voteCount': 127717}",A family coping with addiction over many years...,31749905.0,USD,25000000.0,USD,13,"['Biography', 'Drama']",1.269996,0.239014,"['Biography', 'Drama']",1


In [4]:
import ast, pandas as pd, numpy as np
import plotly.graph_objects as go

def primary_genre(x):
    try:
        genres = ast.literal_eval(x)
        return genres[0] if genres else "Unknown"
    except:
        return "Unknown"

df["primary_genre"] = df["genres"].apply(primary_genre)
df["budget_M"]      = df["productionBudget"] / 1e6
df["gross_M"]       = df["worldwideGross"] / 1e6
df["runtime_min"]   = df["runtimeSeconds"] / 60
df = df.dropna(subset=["budget_M", "gross_M", "totalStarMeter", "runtime_min"])

top_genres = ["Action", "Drama", "Comedy", "Crime", "Adventure", "Horror", "Animation", "Biography"]
df = df[df["primary_genre"].isin(top_genres)]

genre_colors = {
    "Action":    "#e63946",
    "Drama":     "#457b9d",
    "Comedy":    "#f4a261",
    "Crime":     "#6d6875",
    "Adventure": "#2a9d8f",
    "Horror":    "#1d3557",
    "Animation": "#e9c46a",
    "Biography": "#8ecae6",
}

fig = go.Figure()

for genre in top_genres:
    g = df[df["primary_genre"] == genre]
    fig.add_trace(go.Scatter(
        x=g["budget_M"],
        y=g["gross_M"],
        mode="markers",
        name=genre,
        marker=dict(
            size=g["runtime_min"] / 4,
            color=genre_colors[genre],
            opacity=0.8,
            line=dict(width=1, color="white"),
            sizemode="area",
            sizeref=2. * df["runtime_min"].max() / (60.**2),
            sizemin=4,
        ),
        customdata=np.stack([
            g["primaryTitle"],
            g["runtime_min"].round(0),
            g["totalStarMeter"],
            g["grossBudgetRatio"].round(2),
            g["startYear"].astype(int)
        ], axis=-1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Budget: $%{x:.0f}M<br>"
            "Gross: $%{y:.0f}M<br>"
            "Runtime: %{customdata[1]} min<br>"
            "Star Meter: %{customdata[2]}<br>"
            "ROI Ratio: %{customdata[3]}x<br>"
            "Year: %{customdata[4]}<br>"
            "<extra></extra>"
        )
    ))

# Breakeven line
max_budget = df["budget_M"].max() * 1.1
fig.add_trace(go.Scatter(
    x=[0, max_budget],
    y=[0, max_budget],
    mode="lines",
    name="Breakeven (1:1)",
    line=dict(color="rgba(255,255,255,0.3)", dash="dash", width=1.5),
    hoverinfo="skip",
))

fig.update_layout(
    title=dict(
        text="<b>Production Budget vs Worldwide Gross</b><br><sup>Bubble size = Runtime · Color = Genre · Hover for details</sup>",
        font=dict(size=20, color="white"),
        x=0.5,
        xanchor="center",
    ),
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="white", family="Inter, sans-serif"),
    xaxis=dict(
        title="Production Budget ($M)",
        gridcolor="rgba(255,255,255,0.07)",
        zerolinecolor="rgba(255,255,255,0.15)",
        tickprefix="$",
        ticksuffix="M",
        type="log",
        title_font=dict(size=13),
    ),
    yaxis=dict(
        title="Worldwide Gross ($M)",
        gridcolor="rgba(255,255,255,0.07)",
        zerolinecolor="rgba(255,255,255,0.15)",
        tickprefix="$",
        ticksuffix="M",
        type="log",
        title_font=dict(size=13),
    ),
    legend=dict(
        title="Primary Genre",
        bgcolor="rgba(255,255,255,0.05)",
        bordercolor="rgba(255,255,255,0.15)",
        borderwidth=1,
        font=dict(size=12),
    ),
    hoverlabel=dict(
        bgcolor="#1c1c2e",
        bordercolor="rgba(255,255,255,0.2)",
        font=dict(size=13, color="white"),
    ),
    margin=dict(l=70, r=40, t=100, b=70),
    height=620,
)

fig.show()